In [28]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai

In [29]:
def get_data_root_dir():
    return "../../data/eas_attestations"

In [ ]:
def get_collection_name():
    return "Hackathon-Attestations-Collection-00"

### Read the sampled dataset with Hackathon attestation data

In [ ]:

out_path = f"{get_data_root_dir()}/processed_devfolio_attestations_for_vectorization.jsonl"
data_to_embed=pd.read_json(out_path, lines=True)
data_to_embed.head()
data_to_embed=data_to_embed.to_dict(orient="records")
print(len(data_to_embed))

184
184


### Define the embedding function

In [31]:
response=openai.embeddings.create(
    input="random text",
    model="text-embedding-3-small"
)
response
len(response.data[0].embedding)

1536

In [32]:
def get_embedding(text,model="text-embedding-3-small"):
    response=openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Create Qdrant collection

In [33]:
qdrant_client=QdrantClient(url="http://localhost:6333")

In [ ]:
qdrant_client.create_collection(
    collection_name=,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `Hackathon-Attestations-Collection-00` already exists!"},"time":0.001152542}'

### Embed data

#### Test

In [35]:
pointstruct=PointStruct(
    id=0,
    vector=get_embedding("test text"),
    payload={
        "text": "test text",
        "model": "text-embedding-3-small"
    }
)

In [ ]:
pointstruct

PointStruct(id=0, vector=[-0.025677789002656937, 0.00506477989256382, 0.04459746181964874, -0.06313661485910416, -0.03208582103252411, -0.03707829862833023, 0.00230597797781229, -0.011895193718373775, 0.03427764028310776, 0.002538098022341728, 0.025160275399684906, 0.010038234293460846, -0.016119014471769333, 0.01163643691688776, 0.034734271466732025, 0.04106619581580162, -0.026484500616788864, -0.00604272773489356, -0.02958957850933075, 0.050655413419008255, 0.020898401737213135, 0.012047403492033482, 0.0054719410836696625, -0.013889141380786896, -0.003715821076184511, -0.037443604320287704, -0.03540399298071861, 0.015548228286206722, 0.06167539954185486, -0.06709406524896622, 0.022770581766963005, -0.04724591225385666, 0.009718594141304493, -0.03716962784528732, -0.0016809666994959116, 0.023744724690914154, 0.023562071844935417, 0.029467811807990074, 0.024581877514719963, -0.03786979243159294, -0.03777846693992615, -0.012739958241581917, 0.023318536579608917, 0.015692828223109245, 0.

#### Hackathon Attestation Data

In [ ]:
import json
pointstructs=[]
for i,data in enumerate(data_to_embed):
    print(f'{i} / {len(data_to_embed)}')
    print(json.dumps(data["vectorization_data"], indent=2))
    embedding=get_embedding(data["vectorization_data"])
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data
        )
    )
print(f'Embedded {len(pointstructs)} points')

"{project_attributes: [{trait_type: hackathon_name, value: ETHIndia 2023}, {trait_type: nft_type, value: BUILDER}, {trait_type: team_name, value: Monkster}, {trait_type: project_name, value: Auditron}, {display_type: date, trait_type: project_submission_date, value: 1702172248134}], project_data: {url: https://devfolio.co/projects/auditron-ca42, title: Auditron, subtitle: The AI Audit Ecosystem, technologies: [Ant Design, MetaMask, ethers.js, Express.js, Firestore, React.js, Polygon, IPFS / Filecoin, Push Protocol, Linea], matching_amount_usd: 1, votes: 20, quadratic_votes: 8.944, built_at: ETHIndia 2023 MetaMask, created_on: 10th December 2023, last_edited: 10th December 2023, problem_statement: The AI-Enabled Smart Contract Auditor is a versatile tool with a wide range of applications, making smart contract development and deployment more secure and efficient.\\n\\nSecurity Assurance: Developers can use the auditor during the development phase to proactively identify and address pote

In [37]:
pointstructs

[PointStruct(id=0, vector=[-0.051423899829387665, 0.020823469385504723, 0.04389418289065361, -0.015234549529850483, 0.005891713779419661, 0.005176682490855455, -0.04932258278131485, 0.0026886644773185253, 0.0015084976330399513, 0.021509315818548203, 0.04538261517882347, -0.05375869944691658, -0.02061917446553707, -0.026456167921423912, 0.026529129594564438, 0.0178174190223217, -0.041705310344696045, -0.036247722804546356, -0.0355180986225605, 0.007872642949223518, 0.05335010960698128, 0.047308821231126785, 0.03910784795880318, -0.02896607294678688, 0.0011327413376420736, -0.04532424360513687, -0.024748846888542175, 0.011134062893688679, 0.027842452749609947, -0.0024442404974251986, 0.047133710235357285, -0.04538261517882347, 0.02590165287256241, -0.014935404062271118, -0.004571094643324614, 0.015278327278792858, 0.006752670276910067, -0.05501365289092064, -0.003569685621187091, -0.0032924283295869827, -0.0030334119219332933, -0.06624986231327057, -0.02836778201162815, 0.032862264662981

### Write embedded data to qdrant

In [38]:
qdrant_client.upsert(
    collection_name="Hackathon-Attestations-Collection-00",
    wait=True,
    points=pointstructs
)
print(f'Upserted {len(pointstructs)} points')

Upserted 184 points


### Define a function for data retrieval

In [41]:
def retrieve_data(query,k=5):
    query_embedding=get_embedding(query)
    results=qdrant_client.query_points(
        collection_name="Hackathon-Attestations-Collection-00",
        query=query_embedding,
        limit=k
    )
    return results

### Test Retrieval

In [42]:
retrieve_data("Do you have examples of projects that use attestations?",k=10).points

[ScoredPoint(id=122, version=0, score=0.44510758, payload={'id': '0xf5165bd2a551cba86113ad11e7320f4a85479188adcb3c6f63ea5be79d93c00b', 'attester': '0x3Ce7b2b2a9F3C27aFa6EC511679f606412fb497b', 'recipient': '0xEb26E60BBCdBd93251549902E799FD62D65fff76', 'schema': '0x364a59df1d48d4b6c0f8f0c1176504b252bce5ce57e0d1ca75b1bf70c2f0ec14', 'block_number': 214124365, 'blockchain_name': 'arbitrum', 'decoded_data': {'nft_metadata_ipfs_url': 'https://ipfs.io/ipfs/QmcmTgmDwpEjQ3ZAGP8uj2hFn7TU4pYpkJwTRbX6pAGDX5', 'user_uuid': '9c8918363aa3481cba767a9a56271ee6', 'credential_uuid': 'b1426e6a8b7d4867924f588e95eb70b1', 'hackathon_uuid': 'becdb269b9ea4e708c7d96329563e478', 'user_hackathon_credential_uuid': 'ba156e236002429592585c99ed0de648', 'nft_contract_address': '0xe34494de41383fbad7d1cdba6730d0e943425701'}, 'vectorization_data': "{project_attributes: [{trait_type: hackathon_name, value: ETHIndia 2023}, {trait_type: nft_type, value: BUILDER}, {trait_type: team_name, value: Actalink}, {trait_type: projec